# Learning ReAct (Reasoning + Acting) with `smolagents`

This interactive tutorial demonstrates how to build an autonomous agent using the **ReAct pattern** (Reasoning + Acting) using Hugging Face's `smolagents` framework.

### What is ReAct?
The **ReAct pattern** enables an LLM to solve complex, multi-step problems by looping through:
1. **Thought (Reasoning):** The model analyzes the current state and decides what step to take next.
2. **Action (Acting):** The model executes an action, such as performing a web search or executing Python code.
3. **Observation:** The model receives the output of the action and uses it to inform the next thought.

--- 
## Step 1: Install Required Packages
We need `smolagents` for agent orchestration and `python-dotenv` for managing environment variables.

In [ ]:
!pip install -q smolagents python-dotenv

--- 
## Step 2: Set Up Environment Variables
We prompt for your OpenAI API Key securely so that `OpenAIServerModel` can authenticate.

In [ ]:
import os
from getpass import getpass
from dotenv import load_dotenv

# Load local environment file if available
load_dotenv()

# Prompt for key if not present in environment
if "OPENAI_API_KEY" not in os.environ or not os.environ["OPENAI_API_KEY"]:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API Key: ")

--- 
## Step 3: Instantiate Tools (The "Action" Space)
Tools give agents capabilities to interact with external environments. Here, we equip the agent with `DuckDuckGoSearchTool` to query real-time web search results.

In [ ]:
from smolagents import DuckDuckGoSearchTool

# 1. Initialize search tool
search_tool = DuckDuckGoSearchTool()

--- 
## Step 4: Configure the Model (The Reasoning Engine)
We set up `OpenAIServerModel` using `gpt-4o-mini`. This handles the logical decomposition of tasks and decision-making.

In [ ]:
from smolagents import OpenAIServerModel

# 2. Define the AI model
model = OpenAIServerModel("gpt-4o-mini")

--- 
## Step 5: Construct the `CodeAgent` (The ReAct Loop)
`CodeAgent` generates Python executable blocks at each iteration. We grant explicit permission to import `math` and `re` for post-processing.

In [ ]:
from smolagents import CodeAgent

# 3. Assemble CodeAgent with tools and authorized libraries
agent = CodeAgent(
    tools=[search_tool], 
    model=model,
    additional_authorized_imports=["math", "re"]
)

--- 
## Step 6: Define Problem & Run ReAct Execution Loop
Now we pass a multi-step prompt that requires:
1. Searching external price data.
2. Extracting numerical prices from text.
3. Applying custom financial calculation and conditional logic.

In [ ]:
# 4. Multi-step prompt with discount logic
prompt = (
    "Search for the current price of a 128GB iPhone 15 on Amazon India. "
    "Once you have found the price, calculate the final billing amount using these rules:\n"
    "1. Calculate a 10% instant discount based on the retrieved price.\n"
    "2. If the discount amount is greater than ₹1,500, cap the discount at exactly ₹1,500.\n"
    "3. Subtract the discount from the original price to find the final billing amount.\n"
    "Print out the initial price, the calculated discount applied, and your final billing amount."
)

# 5. Run the agent and observe step-by-step logs
response = agent.run(prompt)
print("\n--- Final Result ---")
print(response)